# Behavior Intermediate Representation (IR) Builder

## Enhanced Pipeline

This notebook implements the **Behavior IR** layer — a structured intermediate representation that sits between raw CAPEv2 logs and the LLM signature-generation step.

### Why an IR?
- Raw logs are noisy and large; feeding them directly to the LLM is unreliable.
- An IR compresses thousands of API calls into a handful of typed, scored behavior nodes.
- The LLM then performs a narrow, well-defined task: **IR → detection rule**.

### Pipeline
```
CAPE JSON  →  Event Normalizer  →  Noise Reducer  →  Behavior IR  →  LLM
               (parse events)      (motif collapse)  (JSON struct)     (sig gen)
```

### IR Schema
```json
{
  "sha256": "abc...",
  "behaviors": [
    {"type": "persistence_registry_runkey", "evidence": ["HKCU\\...\\Run"], "score": 0.8},
    {"type": "dns_c2_beacon",               "evidence": ["domain: x.yz"],   "score": 0.7}
  ],
  "iocs": {"domains": [...], "ips": [...], "files_dropped": [...], "mutexes": [...]},
  "process_graph": {"nodes": [...], "edges": [...]}
}
```

**Input**: CAPEv2 JSON reports in `data/public_small_reports/public_small_reports/`  
**Output**: `data/processed/behavior_ir.json`

In [1]:
# Fix pandas installation
# After running this cell, you MUST restart the kernel!

import subprocess
import sys

# Uninstall pandas completely
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "pandas"], check=False)

# Install a stable pandas 2.x version (compatible with streamlit)
subprocess.run([sys.executable, "-m", "pip", "install", "--no-cache-dir", "pandas==2.2.3"], check=True)

print("\n" + "="*60)
print("IMPORTANT: Restart the kernel now before running other cells!")
print("In VS Code: Click 'Restart' button or Ctrl+Shift+P -> 'Notebook: Restart Kernel'")
print("="*60)


IMPORTANT: Restart the kernel now before running other cells!
In VS Code: Click 'Restart' button or Ctrl+Shift+P -> 'Notebook: Restart Kernel'


In [2]:
# -----------------------------------------------------------
# SECTION 1: Imports and Configuration
# -----------------------------------------------------------
import json
import os
import re
import glob
import warnings
from pathlib import Path
from collections import Counter, defaultdict
from typing import List, Dict, Any

import pandas as pd
from tqdm import tqdm

warnings.filterwarnings("ignore")

# Paths
PROJECT_ROOT  = Path("../").resolve()
DATA_PATH     = PROJECT_ROOT / "data" / "public_small_reports" / "public_small_reports"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print(f"Data path    : {DATA_PATH}")
print(f"Processed dir: {PROCESSED_DIR}")

Data path    : C:\Users\USER\Desktop\S7\malware_analysis\Automated-Dynamic-Analysis-Signature-Generation\data\public_small_reports\public_small_reports
Processed dir: C:\Users\USER\Desktop\S7\malware_analysis\Automated-Dynamic-Analysis-Signature-Generation\data\processed


---
## Section 2: Event Normalizer

Parse every CAPE JSON report into a flat list of typed events.

**Key adaptation for AVAST-CTU dataset:**
- APIs are fully-qualified (`kernel32.dll.WriteProcessMemory`), so we strip the DLL prefix
  before matching against known suspicious API sets.
- There is no `network` or `processes` section — network behavior is inferred from
  network-related API calls (InternetOpen, HttpSendRequest, WinHttpConnect, WSAConnect, etc.).
- Process creation info comes from `executed_commands`.

| Category | Event types | Source |
|---|---|---|
| Process | `process_create` | `summary.executed_commands` |
| File | `file_write`, `file_delete`, `file_read` | `summary.write_files`, etc. |
| Registry | `reg_write`, `reg_delete`, `reg_read` | `summary.write_keys`, etc. |
| API (injection) | `api_injection` | `resolved_apis` matching injection set |
| API (hook) | `api_hook` | `resolved_apis` matching hook set |
| API (network) | `api_network` | `resolved_apis` matching network set |
| API (crypto) | `api_crypto` | `resolved_apis` matching crypto set |
| API (persistence) | `api_persistence` | `resolved_apis` matching persistence set |
| Service | `service_create`, `service_start` | `summary.created/started_services` |

In [3]:
# -----------------------------------------------------------
# SECTION 2: Event Normalizer
# -----------------------------------------------------------

# ---- API classification sets (function name only, no DLL prefix) ----

INJECTION_APIS = {
    "WriteProcessMemory", "VirtualAllocEx", "CreateRemoteThread",
    "NtWriteVirtualMemory", "NtCreateThreadEx", "RtlCreateUserThread",
    "SetThreadContext", "QueueUserAPC", "NtQueueApcThread",
    "NtMapViewOfSection", "NtUnmapViewOfSection",
}

HOOK_APIS = {
    "SetWindowsHookExA", "SetWindowsHookExW",
    "GetAsyncKeyState", "GetKeyState",
    "SetWinEventHook",
}

NETWORK_APIS = {
    # WinINet
    "InternetOpenA", "InternetOpenW",
    "InternetConnectA", "InternetConnectW",
    "HttpSendRequestA", "HttpSendRequestW",
    "HttpSendRequestExA", "HttpSendRequestExW",
    "InternetReadFile", "InternetReadFileExA", "InternetReadFileExW",
    "InternetOpenUrlA", "InternetOpenUrlW",
    "InternetWriteFile",
    "InternetCrackUrlA", "InternetCrackUrlW",
    # WinHTTP
    "WinHttpOpen", "WinHttpConnect", "WinHttpOpenRequest",
    "WinHttpSendRequest", "WinHttpReceiveResponse",
    "WinHttpReadData", "WinHttpQueryHeaders",
    # Winsock
    "WSASocketW", "WSASocketA", "WSAConnect",
    "WSASend", "WSARecv",
    "connect", "send", "recv", "socket",
    "sendto", "recvfrom",
    # URL download
    "URLDownloadToFileA", "URLDownloadToFileW",
    "URLDownloadToCacheFileA", "URLDownloadToCacheFileW",
}

CRYPTO_APIS = {
    "CryptAcquireContextW", "CryptAcquireContextA",
    "CryptEncrypt", "CryptDecrypt",
    "CryptHashData", "CryptDeriveKey",
    "CryptGenKey", "CryptImportKey", "CryptExportKey",
    "CryptCreateHash", "CryptDestroyHash",
    "BCryptEncrypt", "BCryptDecrypt",
}

PERSISTENCE_APIS = {
    "CreateServiceA", "CreateServiceW",
    "StartServiceA", "StartServiceW",
    "RegSetValueExA", "RegSetValueExW",
}

PRIVILEGE_APIS = {
    "AdjustTokenPrivileges", "OpenProcessToken",
    "LookupPrivilegeValueA", "LookupPrivilegeValueW",
    "ImpersonateLoggedOnUser", "DuplicateTokenEx",
}

# Registry Run-key path patterns (persistence)
RUN_KEY_PATTERNS = [
    re.compile(r"\\CurrentVersion\\Run(Once|Services)?\\", re.IGNORECASE),
    re.compile(r"\\CurrentVersion\\Explorer\\Shell Folders", re.IGNORECASE),
    re.compile(r"\\CurrentVersion\\Winlogon", re.IGNORECASE),
]

# Task Scheduler patterns
SCHTASK_PATTERNS = [
    re.compile(r"schtasks", re.IGNORECASE),
    re.compile(r"\\Schedule\\TaskCache", re.IGNORECASE),
]

# Executable file extensions (file dropper)
EXECUTABLE_EXTS = {".exe", ".dll", ".bat", ".ps1", ".vbs", ".scr", ".com", ".pif", ".cmd", ".js", ".wsf"}


def strip_dll_prefix(fq_api: str) -> str:
    """Extract function name from 'dll.FunctionName' format."""
    parts = fq_api.rsplit(".", 1)
    return parts[-1] if len(parts) == 2 else fq_api


def normalize_events(report: dict) -> List[Dict[str, Any]]:
    """
    Parse a single CAPEv2 report dict into a flat list of typed events.
    Handles the AVAST-CTU dataset format where APIs are fully-qualified
    and no network/processes sections exist.
    """
    events: List[Dict[str, Any]] = []
    behavior = report.get("behavior", {})
    summary  = behavior.get("summary", {})

    # ---- Process events (from executed_commands) ----
    for cmd in summary.get("executed_commands", []):
        if isinstance(cmd, str) and cmd.strip():
            events.append({"type": "process_create", "value": cmd.strip(), "category": "process"})

    # ---- File events ----
    for f in summary.get("write_files", []):
        events.append({"type": "file_write", "value": f, "category": "file"})
    for f in summary.get("delete_files", []):
        events.append({"type": "file_delete", "value": f, "category": "file"})
    for f in summary.get("read_files", []):
        events.append({"type": "file_read", "value": f, "category": "file"})

    # ---- Registry events ----
    for k in summary.get("write_keys", []):
        events.append({"type": "reg_write", "value": k, "category": "registry"})
    for k in summary.get("delete_keys", []):
        events.append({"type": "reg_delete", "value": k, "category": "registry"})
    for k in summary.get("read_keys", []):
        events.append({"type": "reg_read",   "value": k, "category": "registry"})

    # ---- Service events ----
    for svc in summary.get("created_services", []):
        if isinstance(svc, str) and svc.strip():
            events.append({"type": "service_create", "value": svc.strip(), "category": "service"})
    for svc in summary.get("started_services", []):
        if isinstance(svc, str) and svc.strip():
            events.append({"type": "service_start", "value": svc.strip(), "category": "service"})

    # ---- Classify resolved APIs (strip DLL prefix first) ----
    for fq_api in summary.get("resolved_apis", []):
        func = strip_dll_prefix(fq_api)
        if func in INJECTION_APIS:
            events.append({"type": "api_injection", "value": fq_api, "category": "api"})
        if func in HOOK_APIS:
            events.append({"type": "api_hook", "value": fq_api, "category": "api"})
        if func in NETWORK_APIS:
            events.append({"type": "api_network", "value": fq_api, "category": "api"})
        if func in CRYPTO_APIS:
            events.append({"type": "api_crypto", "value": fq_api, "category": "api"})
        if func in PERSISTENCE_APIS:
            events.append({"type": "api_persistence", "value": fq_api, "category": "api"})
        if func in PRIVILEGE_APIS:
            events.append({"type": "api_privilege", "value": fq_api, "category": "api"})

    return events


# Quick test
json_files = sorted(glob.glob(str(DATA_PATH / "*.json")))
print(f"Found {len(json_files)} JSON reports")

if json_files:
    with open(json_files[0]) as fh:
        sample = json.load(fh)
    evts = normalize_events(sample)
    print(f"\nEvents in first report: {len(evts)}")
    evt_types = Counter(e["type"] for e in evts)
    for t, c in evt_types.most_common():
        print(f"  {t}: {c}")

Found 48976 JSON reports

Events in first report: 143
  reg_read: 70
  file_read: 22
  process_create: 11
  file_write: 9
  file_delete: 9
  api_network: 7
  api_injection: 5
  api_persistence: 4
  api_privilege: 3
  api_crypto: 2
  reg_write: 1


---
## Section 3: Noise Reducer & Behavior Motif Detector

A **behavior motif** is a higher-level behavioral pattern derived from co-occurrence of low-level events.
Each motif now includes a MITRE ATT&CK mapping.

| Motif | Detection rule | MITRE |
|---|---|---|
| `persistence_registry_runkey` | reg_write to Run/RunOnce key | T1547.001 |
| `persistence_scheduled_task` | schtasks in commands or TaskCache registry | T1053.005 |
| `persistence_service` | service_create or CreateService API | T1543.003 |
| `process_injection` | VirtualAllocEx + WriteProcessMemory + CreateRemoteThread | T1055 |
| `file_dropper` | file_write of executable extension | T1105 |
| `network_c2_communication` | network API calls (InternetOpen, WinHttp, Winsock) | T1071 |
| `keylogger_hook` | SetWindowsHookEx / GetAsyncKeyState | T1056.001 |
| `defense_evasion_crypto` | CryptEncrypt / CryptDecrypt etc. | T1027 |
| `privilege_escalation` | AdjustTokenPrivileges / ImpersonateLoggedOnUser | T1134 |
| `credential_access` | Reads from LSASS path or SAM registry hive | T1003 |
| `discovery_commands` | whoami, ipconfig, tasklist, systeminfo in commands | T1082 |

In [4]:
# -----------------------------------------------------------
# SECTION 3: Noise Reducer & Behavior Motif Detector
# -----------------------------------------------------------

# Discovery commands commonly used for system reconnaissance
DISCOVERY_CMDS = {"whoami", "ipconfig", "tasklist", "systeminfo", "net", "netstat",
                  "nslookup", "ping", "arp", "route", "wmic", "hostname", "ver"}


def deduplicate_events(events: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    """Remove exact duplicate events (same type + value). Preserves insertion order."""
    seen = set()
    unique: List[Dict[str, Any]] = []
    for e in events:
        key = (e["type"], e["value"])
        if key not in seen:
            seen.add(key)
            unique.append(e)
    return unique


def detect_motifs(events: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    """
    Collapse de-duplicated events into scored behavior motifs.
    Each motif includes MITRE ATT&CK technique IDs.
    """
    by_type: Dict[str, List[str]] = defaultdict(list)
    for e in events:
        by_type[e["type"]].append(e["value"])

    motifs: List[Dict[str, Any]] = []

    # ---- Persistence: Registry Run key ----
    run_keys = [v for v in by_type["reg_write"]
                if any(pat.search(v) for pat in RUN_KEY_PATTERNS)]
    if run_keys:
        motifs.append({
            "type": "persistence_registry_runkey",
            "evidence": run_keys[:5],
            "score": min(0.6 + 0.1 * len(run_keys), 1.0),
            "mitre": ["T1547.001"],
        })

    # ---- Persistence: Scheduled Task ----
    schtask_cmds = [v for v in by_type["process_create"]
                    if any(pat.search(v) for pat in SCHTASK_PATTERNS)]
    schtask_keys = [v for v in by_type["reg_write"]
                    if any(pat.search(v) for pat in SCHTASK_PATTERNS)]
    if schtask_cmds or schtask_keys:
        motifs.append({
            "type": "persistence_scheduled_task",
            "evidence": (schtask_cmds + schtask_keys)[:5],
            "score": 0.75,
            "mitre": ["T1053.005"],
        })

    # ---- Persistence: Service creation ----
    svc_evidence = by_type["service_create"] + [
        v for v in by_type["api_persistence"]
        if "CreateService" in v
    ]
    if svc_evidence:
        motifs.append({
            "type": "persistence_service",
            "evidence": list(set(svc_evidence))[:5],
            "score": 0.70,
            "mitre": ["T1543.003"],
        })

    # ---- Process injection (from API calls) ----
    inj_apis = by_type["api_injection"]
    if inj_apis:
        unique_inj = list(set(strip_dll_prefix(a) for a in inj_apis))
        # Higher score when multiple injection primitives co-occur
        score = min(0.3 * len(unique_inj), 1.0)
        motifs.append({
            "type": "process_injection",
            "evidence": unique_inj[:5],
            "score": round(score, 2),
            "mitre": ["T1055"],
        })

    # ---- File dropper ----
    dropped = [v for v in by_type["file_write"]
               if Path(v).suffix.lower() in EXECUTABLE_EXTS]
    if dropped:
        motifs.append({
            "type": "file_dropper",
            "evidence": dropped[:5],
            "score": min(0.4 + 0.1 * len(dropped), 1.0),
            "mitre": ["T1105"],
        })

    # ---- Network C2 communication (from API calls) ----
    net_apis = by_type["api_network"]
    if net_apis:
        unique_net = list(set(strip_dll_prefix(a) for a in net_apis))
        # Score based on diversity of network APIs (more diverse = more suspicious)
        score = min(0.3 + 0.07 * len(unique_net), 1.0)
        motifs.append({
            "type": "network_c2_communication",
            "evidence": unique_net[:8],
            "score": round(score, 2),
            "mitre": ["T1071"],
        })

    # ---- Keylogger / hook ----
    hook_apis = by_type["api_hook"]
    if hook_apis:
        unique_hook = list(set(strip_dll_prefix(a) for a in hook_apis))
        motifs.append({
            "type": "keylogger_hook",
            "evidence": unique_hook,
            "score": 0.80,
            "mitre": ["T1056.001"],
        })

    # ---- Defense evasion: encryption ----
    crypto_apis = by_type["api_crypto"]
    if crypto_apis:
        unique_crypto = list(set(strip_dll_prefix(a) for a in crypto_apis))
        motifs.append({
            "type": "defense_evasion_crypto",
            "evidence": unique_crypto[:5],
            "score": min(0.3 + 0.1 * len(unique_crypto), 0.8),
            "mitre": ["T1027"],
        })

    # ---- Privilege escalation ----
    priv_apis = by_type["api_privilege"]
    if priv_apis:
        unique_priv = list(set(strip_dll_prefix(a) for a in priv_apis))
        motifs.append({
            "type": "privilege_escalation",
            "evidence": unique_priv,
            "score": 0.70,
            "mitre": ["T1134"],
        })

    # ---- Credential access: LSASS / SAM paths ----
    cred_paths = [v for v in (by_type["file_read"] + by_type["reg_read"])
                  if re.search(r"(lsass|\\sam$|\\sam\\|security|ntds\.dit|credential|vault)",
                               v, re.IGNORECASE)]
    if cred_paths:
        motifs.append({
            "type": "credential_access",
            "evidence": cred_paths[:5],
            "score": 0.80,
            "mitre": ["T1003"],
        })

    # ---- Discovery: system reconnaissance commands ----
    discovery_cmds = []
    for cmd in by_type["process_create"]:
        base = cmd.strip().strip('"').split("\\")[-1].split(".")[0]
        tokens = base.split()
        if not tokens:
            continue
        exe = tokens[0].lower()
        if exe in DISCOVERY_CMDS:
            discovery_cmds.append(cmd[:80])
    if discovery_cmds:
        motifs.append({
            "type": "discovery_reconnaissance",
            "evidence": discovery_cmds[:5],
            "score": min(0.3 + 0.1 * len(discovery_cmds), 0.8),
            "mitre": ["T1082", "T1016"],
        })

    # ---- Heavy registry activity ----
    if len(by_type["reg_write"]) >= 10:
        motifs.append({
            "type": "heavy_registry_modification",
            "evidence": [f"{len(by_type['reg_write'])} registry writes"],
            "score": round(min(0.2 + 0.01 * len(by_type["reg_write"]), 0.7), 2),
            "mitre": ["T1112"],
        })

    # Sort motifs by score descending
    motifs.sort(key=lambda m: m["score"], reverse=True)
    return motifs


# Smoke-test
if json_files:
    evts    = normalize_events(sample)
    deduped = deduplicate_events(evts)
    motifs  = detect_motifs(deduped)
    print(f"Events: {len(evts)}  ->  after dedup: {len(deduped)}")
    print(f"\nDetected motifs ({len(motifs)}):")
    for m in motifs:
        mitre_str = ", ".join(m["mitre"])
        print(f"  [{m['score']:.2f}] {m['type']}  [{mitre_str}]  evidence={m['evidence'][:2]}")

Events: 143  ->  after dedup: 143

Detected motifs (9):
  [1.00] process_injection  [T1055]  evidence=['CreateRemoteThread', 'NtUnmapViewOfSection']
  [0.79] network_c2_communication  [T1071]  evidence=['InternetReadFile', 'InternetConnectA']
  [0.75] persistence_scheduled_task  [T1053.005]  evidence=['"C:\\Windows\\system32\\schtasks.exe" /create /tn {9E554C6E-7D11-4FC0-AFA1-FF49DBC84412} /tr "\\"C:\\Users\\comp\\AppData\\Roaming\\Microsoft\\Tiafuzdii\\ikkzowxr.exe\\"" /sc HOURLY /mo 5 /F', '"C:\\Windows\\system32\\schtasks.exe" /create /tn {9028B79B-9BA9-4F9B-A9A3-7A9DDC73B07E} /tr "cmd.exe /C \\"start /MIN C:\\Windows\\system32\\cscript.exe //E:javascript \\"C:\\Users\\comp\\ikkzowxr.wpl\\"\\"" /sc WEEKLY /D TUE /ST 12:00:00 /F']
  [0.70] file_dropper  [T1105]  evidence=['C:\\Users\\comp\\AppData\\Roaming\\Microsoft\\Tiafuzdii\\ikkzowxr.exe', 'C:\\Users\\comp\\AppData\\Local\\Temp\\00003D128A7EB859F65F.exe']
  [0.70] persistence_registry_runkey  [T1547.001]  evidence=['HKEY_CURRENT_

---
## Section 4: IR Builder

Assemble the final compact IR object for each report, including:
- `behaviors`: scored motif list with MITRE mappings
- `iocs`: concrete Indicators of Compromise
- `api_profile`: summary of the API call landscape
- `process_commands`: executed command lines

In [5]:
# -----------------------------------------------------------
# SECTION 4: IR Builder
# -----------------------------------------------------------

def extract_iocs(events: List[Dict[str, Any]], report: dict) -> Dict[str, List[str]]:
    """Collect concrete IOC values from de-duplicated events."""
    by_type: Dict[str, List[str]] = defaultdict(list)
    for e in events:
        by_type[e["type"]].append(e["value"])

    summary = report.get("behavior", {}).get("summary", {})
    mutexes = summary.get("mutexes", [])

    # Dropped executables (from file writes)
    exe_drops = [v for v in by_type["file_write"]
                 if Path(v).suffix.lower() in EXECUTABLE_EXTS]

    return {
        "files_dropped":   list(set(exe_drops))[:20],
        "files_written":   list(set(by_type["file_write"]))[:20],
        "registry_writes": list(set(by_type["reg_write"]))[:20],
        "mutexes":         list(set(m for m in mutexes if isinstance(m, str)))[:20],
        "network_apis":    list(set(strip_dll_prefix(a) for a in by_type["api_network"]))[:20],
        "services":        list(set(by_type["service_create"] + by_type["service_start"]))[:10],
    }


def build_api_profile(report: dict) -> Dict[str, Any]:
    """Build a compact summary of the API call landscape."""
    apis = report.get("behavior", {}).get("summary", {}).get("resolved_apis", [])
    stripped = [strip_dll_prefix(a) for a in apis]
    freq = Counter(stripped)
    return {
        "total":  len(apis),
        "unique": len(set(stripped)),
        "top_10": [{"api": api, "count": cnt} for api, cnt in freq.most_common(10)],
    }


def build_ir(filepath: str, sha256: str = "") -> Dict[str, Any]:
    """
    Build the complete Behavior IR for a single CAPEv2 report file.

    Returns IR dict with schema:
      sha256, behaviors (with mitre), iocs, api_profile, process_commands
    """
    if not sha256:
        sha256 = Path(filepath).stem

    try:
        with open(filepath, "r", encoding="utf-8", errors="ignore") as fh:
            report = json.load(fh)
    except (json.JSONDecodeError, IOError) as err:
        return {"sha256": sha256, "error": str(err),
                "behaviors": [], "iocs": {}, "api_profile": {}, "process_commands": []}

    # Step 1: normalize to typed events
    events  = normalize_events(report)
    # Step 2: remove duplicates
    deduped = deduplicate_events(events)
    # Step 3: detect motifs
    motifs  = detect_motifs(deduped)
    # Step 4: extract IOCs
    iocs    = extract_iocs(deduped, report)
    # Step 5: API profile
    api_prof = build_api_profile(report)
    # Step 6: executed commands
    cmds = report.get("behavior", {}).get("summary", {}).get("executed_commands", [])

    return {
        "sha256":           sha256,
        "behaviors":        motifs,
        "iocs":             iocs,
        "api_profile":      api_prof,
        "process_commands": [c for c in cmds if isinstance(c, str)][:20],
    }


# Demo on the first report
if json_files:
    ir = build_ir(json_files[0])
    print("Sample IR:")
    print(json.dumps(ir, indent=2))

Sample IR:
{
  "sha256": "00003d128a7eb859f65f5780d8fa2b5e52d472665678bf6e388e857fbaed773a",
  "behaviors": [
    {
      "type": "process_injection",
      "evidence": [
        "CreateRemoteThread",
        "NtUnmapViewOfSection",
        "NtMapViewOfSection",
        "VirtualAllocEx",
        "WriteProcessMemory"
      ],
      "score": 1.0,
      "mitre": [
        "T1055"
      ]
    },
    {
      "type": "network_c2_communication",
      "evidence": [
        "InternetReadFile",
        "InternetConnectA",
        "InternetWriteFile",
        "InternetCrackUrlA",
        "InternetOpenA",
        "InternetOpenUrlA",
        "HttpSendRequestA"
      ],
      "score": 0.79,
      "mitre": [
        "T1071"
      ]
    },
    {
      "type": "persistence_scheduled_task",
      "evidence": [
        "\"C:\\Windows\\system32\\schtasks.exe\" /create /tn {9E554C6E-7D11-4FC0-AFA1-FF49DBC84412} /tr \"\\\"C:\\Users\\comp\\AppData\\Roaming\\Microsoft\\Tiafuzdii\\ikkzowxr.exe\\\"\" /sc HOURL

---
## Section 5: Batch IR Generation

Build IR for every available CAPEv2 report and persist the results.

In [6]:
# -----------------------------------------------------------
# SECTION 5: Batch IR Generation
# -----------------------------------------------------------

# Load label mapping to embed family info
labels_path = PROJECT_ROOT / "data" / "public_labels.csv"
hash_to_family: Dict[str, str] = {}
hash_to_type:   Dict[str, str] = {}
if labels_path.exists():
    df_labels = pd.read_csv(labels_path)
    hash_to_family = dict(zip(df_labels["sha256"], df_labels["classification_family"]))
    hash_to_type   = dict(zip(df_labels["sha256"], df_labels["classification_type"]))
    print(f"Loaded labels: {len(df_labels)} entries")
else:
    print("WARNING: Labels CSV not found; all samples will be 'unknown'")

ir_records: List[Dict[str, Any]] = []

print(f"\nBuilding IR for {len(json_files)} reports...")
for fpath in tqdm(json_files, desc="Building IR"):
    sha = Path(fpath).stem
    ir  = build_ir(fpath, sha256=sha)
    ir["family"] = hash_to_family.get(sha, "unknown")
    ir["type"]   = hash_to_type.get(sha,   "unknown")
    ir_records.append(ir)

print(f"\nBuilt {len(ir_records)} IR records")

# Motif stats
motif_counts = Counter(
    m["type"] for ir in ir_records for m in ir.get("behaviors", [])
)
print("\nBehavior motifs across entire dataset:")
for motif, cnt in motif_counts.most_common():
    pct = 100 * cnt / len(ir_records)
    print(f"  {motif:40s}  {cnt:6d}  ({pct:.1f}%)")

# MITRE coverage
all_mitre = set()
for ir in ir_records:
    for b in ir.get("behaviors", []):
        all_mitre.update(b.get("mitre", []))
print(f"\nUnique MITRE ATT&CK techniques covered: {len(all_mitre)}")
print(f"  Techniques: {sorted(all_mitre)}")

Loaded labels: 48976 entries

Building IR for 48976 reports...


Building IR: 100%|██████████| 48976/48976 [11:40<00:00, 69.92it/s] 



Built 48976 IR records

Behavior motifs across entire dataset:
  file_dropper                               40593  (82.9%)
  defense_evasion_crypto                     35619  (72.7%)
  credential_access                          31105  (63.5%)
  heavy_registry_modification                12312  (25.1%)
  privilege_escalation                       11537  (23.6%)
  process_injection                          11492  (23.5%)
  network_c2_communication                    7329  (15.0%)
  persistence_registry_runkey                 6147  (12.6%)
  discovery_reconnaissance                    4685  (9.6%)
  persistence_service                         3164  (6.5%)
  persistence_scheduled_task                  3053  (6.2%)
  keylogger_hook                              2280  (4.7%)

Unique MITRE ATT&CK techniques covered: 13
  Techniques: ['T1003', 'T1016', 'T1027', 'T1053.005', 'T1055', 'T1056.001', 'T1071', 'T1082', 'T1105', 'T1112', 'T1134', 'T1543.003', 'T1547.001']


In [7]:
# -----------------------------------------------------------
# SECTION 5b: Persist the IR
# -----------------------------------------------------------

ir_output_path = PROCESSED_DIR / "behavior_ir.json"

with open(ir_output_path, "w") as fh:
    json.dump(ir_records, fh, indent=2)

print(f"Saved {len(ir_records)} IR records to: {ir_output_path}")

# Summary DataFrame for quick inspection
summary_rows = []
for ir in ir_records:
    mitre_all = set()
    for b in ir.get("behaviors", []):
        mitre_all.update(b.get("mitre", []))
    summary_rows.append({
        "sha256":            ir["sha256"],
        "family":            ir.get("family", "unknown"),
        "type":              ir.get("type", "unknown"),
        "num_behaviors":     len(ir.get("behaviors", [])),
        "top_behavior":      ir["behaviors"][0]["type"] if ir.get("behaviors") else "none",
        "top_score":         ir["behaviors"][0]["score"] if ir.get("behaviors") else 0.0,
        "mitre_techniques":  ",".join(sorted(mitre_all)),
        "num_mitre":         len(mitre_all),
        "num_files_dropped": len(ir.get("iocs", {}).get("files_dropped", [])),
        "num_registry_writes": len(ir.get("iocs", {}).get("registry_writes", [])),
        "num_mutexes":       len(ir.get("iocs", {}).get("mutexes", [])),
        "has_network_apis":  len(ir.get("iocs", {}).get("network_apis", [])) > 0,
        "num_apis_total":    ir.get("api_profile", {}).get("total", 0),
        "num_apis_unique":   ir.get("api_profile", {}).get("unique", 0),
        "num_commands":      len(ir.get("process_commands", [])),
    })

df_ir = pd.DataFrame(summary_rows)
ir_summary_path = PROCESSED_DIR / "behavior_ir_summary.csv"
df_ir.to_csv(ir_summary_path, index=False)

print(f"Saved summary CSV: {ir_summary_path}")
print(f"\nIR Summary Statistics:")
print(df_ir[["num_behaviors", "top_score", "num_mitre", "num_files_dropped",
             "num_apis_total", "num_commands"]].describe())
print(f"\nSamples with network API activity: {df_ir['has_network_apis'].sum()} / {len(df_ir)}")
print(f"\nSamples per family:")
print(df_ir["family"].value_counts())

Saved 48976 IR records to: C:\Users\USER\Desktop\S7\malware_analysis\Automated-Dynamic-Analysis-Signature-Generation\data\processed\behavior_ir.json
Saved summary CSV: C:\Users\USER\Desktop\S7\malware_analysis\Automated-Dynamic-Analysis-Signature-Generation\data\processed\behavior_ir_summary.csv

IR Summary Statistics:
       num_behaviors     top_score     num_mitre  num_files_dropped  \
count   48976.000000  48976.000000  48976.000000       48976.000000   
mean        3.457122      0.739947      3.552781           1.252083   
std         2.403647      0.253113      2.592901           1.519126   
min         0.000000      0.000000      0.000000           0.000000   
25%         2.000000      0.800000      2.000000           1.000000   
50%         3.000000      0.800000      3.000000           1.000000   
75%         4.000000      0.800000      4.000000           1.000000   
max        11.000000      1.000000     12.000000          20.000000   

       num_apis_total  num_commands  
c

---
## Section 6: Visualize Behavior Motifs

Visualize the distribution of detected motifs across the dataset to verify the IR captures meaningful behavioral diversity.

In [8]:
# -----------------------------------------------------------
# SECTION 6: Visualization
# -----------------------------------------------------------
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

if len(df_ir) > 0 and motif_counts:
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    fig.suptitle("Behavior IR Analysis", fontsize=14, fontweight="bold")

    # Top-left: motif frequency
    labels_plot = [k for k, _ in motif_counts.most_common(12)]
    values_plot = [v for _, v in motif_counts.most_common(12)]
    axes[0, 0].barh(labels_plot[::-1], values_plot[::-1], color="steelblue", edgecolor="black")
    axes[0, 0].set_xlabel("Number of samples")
    axes[0, 0].set_title("Behavior Motifs (frequency)")

    # Top-right: behaviors per sample distribution
    max_beh = df_ir["num_behaviors"].max()
    axes[0, 1].hist(df_ir["num_behaviors"], bins=range(0, max_beh + 2),
                    color="darkorange", edgecolor="black", align="left")
    axes[0, 1].set_xlabel("Number of motifs per sample")
    axes[0, 1].set_ylabel("Count")
    axes[0, 1].set_title("Motifs per Sample Distribution")
    axes[0, 1].axvline(df_ir["num_behaviors"].mean(), color="red", linestyle="--",
                       label=f"Mean: {df_ir['num_behaviors'].mean():.1f}")
    axes[0, 1].legend()

    # Bottom-left: MITRE technique coverage
    mitre_counter = Counter()
    for ir in ir_records:
        for b in ir.get("behaviors", []):
            for t in b.get("mitre", []):
                mitre_counter[t] += 1
    if mitre_counter:
        m_labels = [k for k, _ in mitre_counter.most_common(12)]
        m_values = [v for _, v in mitre_counter.most_common(12)]
        axes[1, 0].barh(m_labels[::-1], m_values[::-1], color="coral", edgecolor="black")
        axes[1, 0].set_xlabel("Number of samples")
        axes[1, 0].set_title("MITRE ATT&CK Technique Coverage")

    # Bottom-right: motifs by family (top 5 families)
    top_families = df_ir["family"].value_counts().head(5).index.tolist()
    family_motif_data = {}
    for ir in ir_records:
        fam = ir.get("family", "unknown")
        if fam in top_families:
            if fam not in family_motif_data:
                family_motif_data[fam] = Counter()
            for b in ir.get("behaviors", []):
                family_motif_data[fam][b["type"]] += 1

    # Stacked bar showing top motifs per family
    top_motifs = [k for k, _ in motif_counts.most_common(6)]
    x = np.arange(len(top_families))
    width = 0.13
    for i, motif in enumerate(top_motifs):
        vals = [family_motif_data.get(fam, {}).get(motif, 0) for fam in top_families]
        axes[1, 1].bar(x + i * width, vals, width, label=motif.replace("_", " ")[:20])
    axes[1, 1].set_xticks(x + width * len(top_motifs) / 2)
    axes[1, 1].set_xticklabels(top_families, rotation=30, ha="right")
    axes[1, 1].set_ylabel("Count")
    axes[1, 1].set_title("Motifs by Malware Family (top 5)")
    axes[1, 1].legend(fontsize=7, loc="upper right")

    plt.tight_layout()
    fig_path = PROCESSED_DIR / "behavior_ir_motifs.png"
    plt.savefig(fig_path, dpi=150)
    plt.show()
    print(f"Saved figure: {fig_path}")
else:
    print("No data to visualize")

Saved figure: C:\Users\USER\Desktop\S7\malware_analysis\Automated-Dynamic-Analysis-Signature-Generation\data\processed\behavior_ir_motifs.png


---
## Section 7: IR -> LLM Prompt Helper

Serialize the compact IR into a structured prompt for the LLM.
The LLM receives *only* the IR — not raw API logs — and generates a **CAPEv2 Python signature class**.

This is the critical interface between the IR pipeline and LLM signature generation.

In [9]:
# -----------------------------------------------------------
# SECTION 7: IR-to-Prompt Serializer
# -----------------------------------------------------------

# MITRE ATT&CK technique descriptions (for enriching prompts)
MITRE_DESCRIPTIONS = {
    "T1547.001": "Boot/Logon Autostart: Registry Run Keys",
    "T1053.005": "Scheduled Task/Job: Scheduled Task",
    "T1543.003": "Create or Modify System Process: Windows Service",
    "T1055":     "Process Injection",
    "T1105":     "Ingress Tool Transfer",
    "T1071":     "Application Layer Protocol (C2)",
    "T1056.001": "Input Capture: Keylogging",
    "T1027":     "Obfuscated Files or Information",
    "T1134":     "Access Token Manipulation",
    "T1003":     "OS Credential Dumping",
    "T1082":     "System Information Discovery",
    "T1016":     "System Network Configuration Discovery",
    "T1112":     "Modify Registry",
}


def ir_to_prompt(ir: Dict[str, Any]) -> str:
    """
    Serialize a Behavior IR object into a compact, structured LLM prompt.
    The LLM receives only this — its job is: IR -> CAPEv2 signature code.
    """
    lines = []
    lines.append("=== BEHAVIOR IR ===")
    lines.append(f"SHA256 : {ir['sha256'][:16]}...")
    if ir.get("family") and ir["family"] != "unknown":
        lines.append(f"Family : {ir['family']}")
    if ir.get("type") and ir["type"] != "unknown":
        lines.append(f"Type   : {ir['type']}")

    # Behaviors with MITRE mapping
    lines.append("\n-- Scored Behavior Motifs --")
    if ir.get("behaviors"):
        for b in ir["behaviors"]:
            ev = "; ".join(str(e) for e in b["evidence"][:3])
            mitre = ", ".join(b.get("mitre", []))
            mitre_desc = " | ".join(
                MITRE_DESCRIPTIONS.get(t, t) for t in b.get("mitre", [])
            )
            lines.append(f"  [{b['score']:.2f}] {b['type']}  [{mitre}]")
            lines.append(f"         ATT&CK: {mitre_desc}")
            lines.append(f"         Evidence: {ev}")
    else:
        lines.append("  (no significant behaviors detected)")

    # IOCs
    iocs = ir.get("iocs", {})
    non_empty_iocs = {k: v for k, v in iocs.items() if v}
    if non_empty_iocs:
        lines.append("\n-- IOCs --")
        for ioc_type, vals in non_empty_iocs.items():
            lines.append(f"  {ioc_type}: {vals[:5]}")

    # API profile
    api_prof = ir.get("api_profile", {})
    if api_prof.get("total"):
        lines.append(f"\n-- API Profile --")
        lines.append(f"  Total: {api_prof['total']}  Unique: {api_prof['unique']}")
        top_apis = [f"{a['api']}({a['count']})" for a in api_prof.get("top_10", [])[:5]]
        lines.append(f"  Top: {', '.join(top_apis)}")

    # Commands
    cmds = ir.get("process_commands", [])
    if cmds:
        lines.append(f"\n-- Executed Commands ({len(cmds)}) --")
        for cmd in cmds[:5]:
            lines.append(f"  > {cmd[:100]}")

    lines.append("\n=== END IR ===")
    return "\n".join(lines)


# Demo on first IR record
if ir_records:
    demo_ir = ir_records[0]
    prompt  = ir_to_prompt(demo_ir)
    print(prompt)
    print(f"\n--- Prompt length: {len(prompt)} chars ---")

=== BEHAVIOR IR ===
SHA256 : 00003d128a7eb859...
Family : Qakbot
Type   : banker

-- Scored Behavior Motifs --
  [1.00] process_injection  [T1055]
         ATT&CK: Process Injection
         Evidence: CreateRemoteThread; NtUnmapViewOfSection; NtMapViewOfSection
  [0.79] network_c2_communication  [T1071]
         ATT&CK: Application Layer Protocol (C2)
         Evidence: InternetReadFile; InternetConnectA; InternetWriteFile
  [0.75] persistence_scheduled_task  [T1053.005]
         ATT&CK: Scheduled Task/Job: Scheduled Task
         Evidence: "C:\Windows\system32\schtasks.exe" /create /tn {9E554C6E-7D11-4FC0-AFA1-FF49DBC84412} /tr "\"C:\Users\comp\AppData\Roaming\Microsoft\Tiafuzdii\ikkzowxr.exe\"" /sc HOURLY /mo 5 /F; "C:\Windows\system32\schtasks.exe" /create /tn {9028B79B-9BA9-4F9B-A9A3-7A9DDC73B07E} /tr "cmd.exe /C \"start /MIN C:\Windows\system32\cscript.exe //E:javascript \"C:\Users\comp\ikkzowxr.wpl\"\"" /sc WEEKLY /D TUE /ST 12:00:00 /F
  [0.70] file_dropper  [T1105]
         ATT

---
## Summary

| Step | Description | Output |
|---|---|---|
| 1. Event Normalizer | CAPE JSON -> typed event list (strips DLL prefix from APIs) | `List[{type, value, category}]` |
| 2. Noise Reducer | remove duplicate events | deduplicated event list |
| 3. Motif Detector | collapse events -> scored behavior motifs with MITRE ATT&CK | `[{type, evidence, score, mitre}]` |
| 4. IOC Extractor | collect concrete IOCs | `{files_dropped, registry_writes, mutexes, network_apis, ...}` |
| 5. API Profile | compact summary of API call landscape | `{total, unique, top_10}` |
| **6. IR Builder** | **assemble full IR** | **`behavior_ir.json`** |
| 7. IR -> Prompt | compact structured LLM prompt | string for any LLM backend |

### Key design decisions:
- **DLL-prefix stripping**: APIs in AVAST-CTU are `kernel32.dll.WriteProcessMemory` — we strip to `WriteProcessMemory` for matching
- **Network from APIs**: No `network` section in dataset — network behavior inferred from WinINet/WinHTTP/Winsock API calls
- **MITRE ATT&CK**: Every motif carries technique IDs, enabling automated ATT&CK mapping reports
- **Scoring**: Heuristic 0-1 scores based on evidence diversity; multiple injection primitives score higher than single ones

**Next step**: Load `data/processed/behavior_ir.json` in `llm_behavior_analysis.ipynb` and generate CAPEv2 Python signature classes.